# Continuous optimization

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/calm.ipynb)

CALM learns a weighted DAG by gradient descent instead of searching over graphs. It first estimates
the moral graph, which leaves only the pairs of variables that can share an edge. It then
minimizes the Gaussian likelihood plus a penalty per edge, under a constraint that the graph is
acyclic. The fit runs on torch, so it needs the `torch` extra.

NOTEARS, the method this line of work started from, minimizes least squares plus an `l1` penalty
under the same kind of constraint. It needs no torch, and the last section shows how its result
depends on the units of the columns.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.17.1` from PyPI;
local kernels use the installed environment. CALM needs torch, which Colab provides; the next cell
installs it with uv in a local kernel that lacks it.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[viz]==0.17.1" pandas

In [2]:
from importlib.util import find_spec

if not IN_COLAB and find_spec("torch") is None:
    !uv pip install --python {sys.executable} --torch-backend=auto "torch>=2.2"

In [3]:
import numpy as np
from IPython.display import display

import andrey
import andrey.metrics as metrics
import andrey.viz
from andrey.data import benchmarks, sample_scm

## Data

`linear_gauss_er` draws an Erdos-Renyi graph with linear mechanisms and Gaussian noise: here 10
variables and 100 samples, ten per variable, the size the 0.7.0 benchmark measured.

In [4]:
seed = 7
dataset = benchmarks.scm("linear_gauss_er", d=10).sample(n=100, seed=seed, scale="standardize")
X, truth = dataset.data, dataset.observed_graph
print(f"{X.shape[0]} observations, {X.shape[1]} variables, seed {seed}")
display(andrey.viz.draw(truth))

100 observations, 10 variables, seed 7


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 464 332" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="464" height="332" fill="#fafbfc"/><line x1="312.82" y1="44.82" x2="242.84" y2="91.09" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.16,39.30 310.06,40.65 315.57,48.99" fill="#3a4658" stroke-linejoin="round"/><line x1="144.93" y1="163.00" x2="409.13" y2="134.59" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.07,133.52 408.60,129.62 409.67,139.56" fill="#3a4658" stroke-linejoin="round"/><line x1="113.24" y1="177.70" x2="42.60" y2="227.83" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.40,171.91 110.35,173.62 116.14,181.78" fill="#3a4658" stroke-linejoin="round"/><line x1="112.82" y1="151.70" x2="42.84" y2="105.43" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.16,157.22 115.57,147.53 110.06,155.87" fill="#3a4658" stroke-linejoin="round"/><line x1="409.32" y1="128.29" x2="244.82" y2="100.43" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.18,129.96 410.16,123.36 408.49,133.22" fill="#3a4658" stroke-linejoin="round"/><line x1="410.18" y1="124.86" x2="144.33" y2="36.24" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.67,128.02 411.76,120.11 408.60,129.60" fill="#3a4658" stroke-linejoin="round"/><line x1="410.11" y1="139.19" x2="344.37" y2="160.40" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.63,136.12 408.58,134.43 411.65,143.95" fill="#3a4658" stroke-linejoin="round"/><line x1="139.66" y1="224.85" x2="218.45" y2="116.84" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="224.34,108.76 214.41,113.89 222.49,119.79" fill="#3a4658" stroke-linejoin="round"/><line x1="142.92" y1="242.40" x2="212.67" y2="287.40" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="221.08,292.82 215.38,283.20 209.96,291.60" fill="#3a4658" stroke-linejoin="round"/><line x1="109.00" y1="235.35" x2="45.00" y2="235.35" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,235.35 109.00,230.35 109.00,240.35" fill="#3a4658" stroke-linejoin="round"/><line x1="212.82" y1="85.57" x2="142.84" y2="39.30" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="221.16,91.09 215.57,81.40 210.06,89.74" fill="#3a4658" stroke-linejoin="round"/><line x1="242.84" y1="105.43" x2="312.82" y2="151.70" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.16,157.22 315.57,147.53 310.06,155.87" fill="#3a4658" stroke-linejoin="round"/><line x1="209.00" y1="98.26" x2="45.00" y2="98.26" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,98.26 209.00,93.26 209.00,103.26" fill="#3a4658" stroke-linejoin="round"/><line x1="112.82" y1="44.82" x2="42.84" y2="91.09" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.16,39.30 110.06,40.65 115.57,48.99" fill="#3a4658" stroke-linejoin="round"/><line x1="210.11" y1="292.81" x2="44.37" y2="239.35" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.63,295.88 211.65,288.05 208.58,297.57" fill="#3a4658" stroke-linejoin="round"/><circle cx="332.00" cy="32.13" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="332.00" y="32.13" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="132.00" cy="164.39" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="164.39" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="432.00" cy="132.13" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="432.00" y="132.13" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominan

## Fit CALM

`andrey.calm` returns a weighted DAG. Its optimizer draws random numbers and is seeded on every
call, so a run repeats on one machine. The fit takes 10 to 30 seconds on a CPU.

In [5]:
out = andrey.calm(X)
print(f"{out.structure.kind}, {len(out.structure.to_edges())} edges, seed {out.metadata['seed']}")
display(andrey.viz.draw(out.structure))

dag, 10 edges, seed 0


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 364 325" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="364" height="325" fill="#fafbfc"/><line x1="110.32" y1="39.55" x2="44.25" y2="62.99" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.75,36.20 108.65,34.83 112.00,44.26" fill="#3a4658" stroke-linejoin="round"/><line x1="242.60" y1="214.65" x2="313.24" y2="164.52" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.40,158.73 310.35,160.44 316.14,168.60" fill="#3a4658" stroke-linejoin="round"/><line x1="245.00" y1="222.18" x2="309.00" y2="222.18" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.00,222.18 309.00,217.18 309.00,227.18" fill="#3a4658" stroke-linejoin="round"/><line x1="309.00" y1="151.21" x2="245.00" y2="151.21" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.00,151.21 309.00,146.21 309.00,156.21" fill="#3a4658" stroke-linejoin="round"/><line x1="309.64" y1="145.80" x2="144.64" y2="105.88" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.36,148.15 310.82,140.94 308.47,150.66" fill="#3a4658" stroke-linejoin="round"/><line x1="144.12" y1="185.23" x2="210.55" y2="159.51" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.88,155.90 208.75,154.85 212.36,164.18" fill="#3a4658" stroke-linejoin="round"/><line x1="141.05" y1="199.26" x2="216.00" y2="276.63" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="222.95,283.81 219.59,273.15 212.41,280.10" fill="#3a4658" stroke-linejoin="round"/><line x1="44.25" y1="71.69" x2="110.32" y2="95.13" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.75,98.48 112.00,90.42 108.65,99.84" fill="#3a4658" stroke-linejoin="round"/><line x1="211.30" y1="141.19" x2="143.70" y2="108.48" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="220.30,145.55 213.47,136.69 209.12,145.69" fill="#3a4658" stroke-linejoin="round"/><line x1="145.00" y1="293.15" x2="209.00" y2="293.15" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,293.15 209.00,288.15 209.00,298.15" fill="#3a4658" stroke-linejoin="round"/><circle cx="132.00" cy="31.85" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="31.85" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="232.00" cy="222.18" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="222.18" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="332.00" cy="151.21" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="332.00" y="151.21" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">2</text><circle cx="132.00" cy="189.92" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="189.92" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">3</text><circle cx="32.00" cy="67.34" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="67.34" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">4</text><circle cx="232.00" cy="151.21" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="151.21" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">5</text><circle cx="132.00" cy="293.15" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="293.15" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">6</text><circle cx="232.00" cy="293.15" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="293.15" t

## Check accuracy

SHD counts errors against the true graph; lower is better. Gaussian data identify the graph only up
to its equivalence class, so `mec_shd` compares the two CPDAGs. The empty graph's SHD is the number
of true edges, so an estimate at or above it has not improved on returning no edges.
One dataset can land on either side: on the 0.7.0 benchmark's five datasets of this size,
CALM's median was 6 errors against the empty graph's 10.

In [6]:
d = truth.n_nodes
empty = andrey.GraphStructure.from_numpy(np.zeros((d, d), dtype=np.uint8), kind="dag")
print("CALM SHD:", metrics.score(out, truth)["mec_shd"])
print("Empty graph SHD:", metrics.score(empty, truth)["mec_shd"])

CALM SHD: 14
Empty graph SHD: 15


## Cost and limits

Every optimizer step computes a determinant and a matrix power of the full weight matrix, so a
step's cost grows with the cube of the variable count. The median fit in the `andrey.calm`
docstring took 18 s at 10 variables and 59 s at 50, on 4 cores. The fit is nonconvex and can stop
at a local optimum, and one dataset can favor any method: change `seed`, `d`, or `n` and rerun.

## NOTEARS

`andrey.notears` minimizes the least-squares loss of a linear model plus an `l1` penalty on the
weights, under the constraint that the weight matrix is acyclic, and returns a weighted DAG. It
draws no random numbers, so a run repeats.

The data follow the simulation of the NOTEARS paper: a linear model on an Erdos-Renyi graph, here
10 variables and 20 edges, with 1,000 samples and noise of the same variance on every variable.
`scale="raw"` keeps the columns as the model generates them.

In [7]:
sem = sample_scm(d=10, n=1000, seed=seed, scale="raw")
print(f"{sem.data.shape[0]} observations, {sem.data.shape[1]} variables, seed {seed}")
print("Standard deviation of each column:", np.round(sem.data.std(axis=0), 1))

1000 observations, 10 variables, seed 7
Standard deviation of each column: [4.3 1.  2.9 3.8 1.9 2.1 1.2 1.  4.  3.1]


### Fit NOTEARS on raw and on standardized columns

Least squares reads the columns in their given units. With equal noise variances, a variable's
variance grows along the causal order, so on these raw columns the fit can read the order from the
variances. The units of measured data are whatever they were recorded in, and
`standardize=True` scales each column to unit variance first, which removes that information.
Each fit takes about a second.

In [8]:
raw = andrey.notears(sem.data)
standardized = andrey.notears(sem.data, standardize=True)
print(f"raw columns: {raw.structure.kind}, {len(raw.structure.to_edges())} edges")
print(f"standardized columns: {len(standardized.structure.to_edges())} edges")
display(andrey.viz.draw(standardized.structure))

raw columns: dag, 20 edges
standardized columns: 11 edges


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 664 240" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="664" height="240" fill="#fafbfc"/><line x1="112.96" y1="83.71" x2="42.76" y2="131.26" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.24,78.10 110.15,79.57 115.76,87.85" fill="#3a4658" stroke-linejoin="round"/><line x1="144.99" y1="70.39" x2="209.01" y2="68.32" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.01,68.00 208.85,63.32 209.17,73.32" fill="#3a4658" stroke-linejoin="round"/><line x1="313.24" y1="125.24" x2="242.60" y2="75.10" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.40,131.02 316.14,121.16 310.35,129.31" fill="#3a4658" stroke-linejoin="round"/><line x1="309.00" y1="138.55" x2="245.00" y2="138.55" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.00,138.55 309.00,133.55 309.00,143.55" fill="#3a4658" stroke-linejoin="round"/><line x1="345.00" y1="138.55" x2="409.00" y2="138.55" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.00,138.55 409.00,133.55 409.00,143.55" fill="#3a4658" stroke-linejoin="round"/><line x1="45.00" y1="138.55" x2="209.00" y2="138.55" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,138.55 209.00,133.55 209.00,143.55" fill="#3a4658" stroke-linejoin="round"/><path d="M 44.94 139.84 Q 331.27 168.46 609.00 172.12" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="619.00,172.25 609.07,167.12 608.94,177.12" fill="#3a4658" stroke-linejoin="round"/><line x1="245.00" y1="67.58" x2="309.00" y2="67.58" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.00,67.58 309.00,62.58 309.00,72.58" fill="#3a4658" stroke-linejoin="round"/><line x1="510.32" y1="95.37" x2="444.25" y2="71.93" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="519.75,98.72 512.00,90.66 508.65,100.09" fill="#3a4658" stroke-linejoin="round"/><line x1="542.68" y1="110.47" x2="613.10" y2="159.31" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="621.32,165.01 615.95,155.20 610.25,163.42" fill="#3a4658" stroke-linejoin="round"/><line x1="409.00" y1="67.58" x2="345.00" y2="67.58" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.00,67.58 409.00,62.58 409.00,72.58" fill="#3a4658" stroke-linejoin="round"/><circle cx="132.00" cy="70.81" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="70.81" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="332.00" cy="138.55" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="332.00" y="138.55" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="32.00" cy="138.55" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="138.55" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">2</text><circle cx="232.00" cy="67.58" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="67.58" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">3</text><circle cx="232.00" cy="138.55" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="138.55" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">4</text><circle cx="532.00" cy="103.06" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="532.00" y="103.06" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">5</text><circle cx="432.00" cy="138.55" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="432.00" y="138.55" text-anchor="mi

### Accuracy on each scale

NOTEARS orients every edge, so `shd` compares each DAG with the true one.

In [9]:
empty = andrey.GraphStructure.from_numpy(np.zeros((10, 10), dtype=np.uint8), kind="dag")
print("NOTEARS SHD, raw columns:", metrics.score(raw, sem.graph)["shd"])
print("NOTEARS SHD, standardized columns:", metrics.score(standardized, sem.graph)["shd"])
print("Empty graph SHD:", metrics.score(empty, sem.graph)["shd"])

NOTEARS SHD, raw columns: 0
NOTEARS SHD, standardized columns: 14
Empty graph SHD: 20


The accuracy on raw columns depends on the units: it is high here because the variances follow the
causal order, and rescaling a column can change the graph. On standardized columns NOTEARS is
modestly better than the empty graph on sparse random graphs, and one dataset can land on either
side. On the 0.17.0 benchmark's 20 datasets of this size, its mean was 2.4 errors on raw columns
and 18.4 on standardized ones, against the empty graph's 20; on scale-free graphs with four edges
per variable, its errors on standardized columns were about equal to the empty graph's. The
acyclicity function takes a matrix exponential of the full weight matrix: the median fit took
about 2 s at 10 variables and 40 s at 50, on one core.